# ESA CCI CMIP7 recipe example

Shows a bundled CMIP7 plugin recipe for a tiny ESA CCI water-vapour dataset.

In [ ]:
import numpy as np
import woodpecker_cmip7_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip7

Create an ESA CCI-like input with names, metadata, and latitude order that need repair.

In [ ]:
source_name = "ESACCI-WATERVAPOUR-L3C-TCWV-meris-005deg-2002-2017-fv3.2.zarr"
dataset = make_cmip7(variable="prw", overrides={"source_name": source_name}, seed=7)
dataset = dataset.isel(lat=slice(None, None, -1))
dataset = dataset.assign_coords(bnds=[0, 1])
dataset["lat_bnds"] = (
    ("lat", "bnds"),
    np.column_stack([dataset["lat"].values - 0.5, dataset["lat"].values + 0.5]),
)

dataset

Load the bundled ESA CCI recipe and inspect its match rules and fix steps.

In [ ]:
recipe = woodpecker.recipe.get("cmip7.esa_cci_water_vapour_zarr")

recipe.model_dump()

In [ ]:
recipe.match.model_dump(), [step.id for step in recipe.steps]

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)
findings.fix_ids

Dry-run previews the repair without changing the dataset.

In [ ]:
result = woodpecker.recipe.apply(dataset, recipe, dry_run=True)

result.stats, result.preview, tuple(dataset.data_vars), tuple(dataset.dims)

Apply the recipe in memory and re-check.

In [ ]:
write = woodpecker.recipe.apply(dataset, recipe, dry_run=False)

(
    write.stats,
    tuple(dataset.data_vars),
    tuple(dataset.dims),
    dataset.attrs["realm"],
    dataset.attrs["branded_variable"],
    float(dataset["lat"].values[0]) < float(dataset["lat"].values[-1]),
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)
bool(recheck)